# Lesson 1
## Embeddings
A way to represent sentences as vectors in an embedding space(vector space).
Semantically similar vectors get placed near to eachother. 

Example:

These two sentences:
* “The dog is running”
* “A puppy is sprinting”

look different as text, but semantically they’re similar.
An embedding model maps both into nearby points in vector space. The cliser teh vectors teh similar the meaning they share.

In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The dog is running in the park",
    "A puppy is sprinting outside",
    "SQL is used for databases"
]

embeddings = model.encode(sentences)

query = model.encode(['A dog is running'])

scores = cosine_similarity(query, embeddings)
print(scores)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[ 0.66139054  0.69718087 -0.0452537 ]]


The 2nd embedded sentence shows the highest cosine_similarity score which means its most semantically similar to the query. 

In [16]:
query.shape #the model represented the sentence as a 384 dimension vector 

(1, 384)

In [17]:
# Test 2: this one shows how keyword search and embeddings search differ

sentences = [
    "The server crashed because memory was exhausted",
    "The RAM overflowed causing the machine to fail",
    "The word memory appears here but unrelated"
]

embeddings = model.encode(sentences)

query = model.encode(["system failed due to low memory"])

scores = cosine_similarity(query, embeddings)

for sentence, score in zip(sentences, scores[0]):
    print(sentence, '->', score)

The server crashed because memory was exhausted -> 0.6742977
The RAM overflowed causing the machine to fail -> 0.6393724
The word memory appears here but unrelated -> 0.39389783


Here the cosine_similarity ran linearly comparing the query embeddings to each of the sentence embeddings but,

* 10 chunks → easy
* 10,000 chunks → slower
* 1,000,000 chunks → needs indexing tricks

Thats why vector databases exist. It avoids brute-force scans by using approximate nearest-neighbor methods.

# Lesson 2
## Vector Databases and Chunking
**They store:**
* vectors
* metadata
* document IDs
* chunk text

**They handle:**
* fast nearest-neighbor lookup
* filtering
* persistence

### Chunking
Diving bigger text paragraphs into smaller size chunks.

In [18]:
def chunk_text(text, chunk_size=500):

    chunks=[]
    start=0

    while start < len(text):
        end   = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += end

    return chunks

In [19]:
text = '''Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When you write a prompt, you are attempting to set up the LLM to predict the right sequence
of tokens. Prompt engineering is the process of designing high-quality prompts that guide
LLMs to produce accurate outputs. This process involves tinkering to find the best prompt,
optimizing prompt length, and evaluating a prompt’s writing style and structure in relation
to the task. In the context of natural language processing and LLMs, a prompt is an input
provided to the model to generate a response or prediction.
LLMs are tuned to follow instructions and are trained on large amounts of data so they can
understand a prompt and generate an answer. But LLMs aren’t perfect; the clearer your
prompt text, the better it is for the LLM to predict the next likely text. Additionally, specific
techniques that take advantage of how LLMs are trained and how LLMs work will help you get
the relevant results from LLMs
Now that we understand what prompt engineering is and what it takes, let’s dive into some
examples of the most important prompting techniques.
General prompting / zero shot
A zero-shot5
prompt is the simplest type of prompt. It only provides a description of a task
and some text for the LLM to get started with. This input could be anything: a question, a
start of a story, or instructions. The name zero-shot stands for ’no examples’.'''

In [20]:
import numpy as np

chunks = chunk_text(text=text)

embeddings = model.encode(chunks)

query = model.encode(['working of a llm'])

scores = cosine_similarity(query, embeddings)[0]

top_indices = np.argsort(scores)[::-1][:2] #top 2 best matches

for i in top_indices:
    print(chunks[i])
    print(scores[i])

Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When 
0.5209081
you write a prompt, you are attempting to set up the LLM to predict the right sequence
of tokens. Prompt engineering is the process of designing high-quality prompts that guide
LLMs to produce accurate outputs. This process involves tinkering to find the best prompt,
optimizing prompt length, and evaluating a prompt’s writing style and structure in relation
to the task. In the context of natural language processing and LLMs, a prompt is an input
provided to the model to generate a res

lets introduce chunking with overlap by modifing out chunk_text method, for that we'll have to add a overlap attribute and a little tweak to the code should do it

In [21]:
def chunk_text_with_overlap(text, chunk_size=500, overlap=10):

    chunks=[]
    start=0

    while start < len(text):
        end   = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [22]:
import numpy as np

chunks = chunk_text_with_overlap(text=text)

embeddings = model.encode(chunks)

query = model.encode(['working of a llm'])

scores = cosine_similarity(query, embeddings)[0]

top_indices = np.argsort(scores)[::-1][:2] 

for i in top_indices:
    print(chunks[i])
    print(scores[i])

Remember how an LLM works; it’s a prediction engine. The model takes sequential text as
an input and then predicts what the following token should be, based on the data it was
trained on. The LLM is operationalized to do this over and over again, adding the previously
predicted token to the end of the sequential text for predicting the following token. The next
token prediction is based on the relationship between what’s in the previous tokens and what
the LLM has seen during its training.
When 
0.5209081
ate a response or prediction.
LLMs are tuned to follow instructions and are trained on large amounts of data so they can
understand a prompt and generate an answer. But LLMs aren’t perfect; the clearer your
prompt text, the better it is for the LLM to predict the next likely text. Additionally, specific
techniques that take advantage of how LLMs are trained and how LLMs work will help you get
the relevant results from LLMs
Now that we understand what prompt engineering is and what it 

Overlapping preserves continuity. Also a thing to notice is that teh score of the 2nd chunk increased which means overlapping helps preserve the continuity and flow over all the chunks.

# Lesson 3
## Real Document QA using LlamaIndex

In [23]:
pip install llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 95.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
Note: you may need to restart the kernel to use updated packages.


In [24]:
pip install llama-index-embeddings-huggingface

Note: you may need to restart the kernel to use updated packages.


In [25]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.6 MB/s eta 0:00:00
  Attempting uninstall: o

In [26]:
!pip install pypdf

In [27]:
!pip install llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00


In [28]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.readers.file import PDFReader
from pathlib import Path

dataset = Path("/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs")

loader = PDFReader()

documents = []

for pdf in dataset.glob("*.pdf"):
    doc = loader.load_data(file=pdf)
    documents.extend(doc)
    
print(documents[0].text[:500])

Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic



In [29]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

index = VectorStoreIndex.from_documents(documents)

retriever = index.as_retriever(similarity_top_k=2)

nodes = retriever.retrieve("What is prompt engineering?")

for i, node in enumerate(nodes):
    print(f"\n--- Chunk {i+1} ---\n")
    print(node.text[:1000])

2026-05-27 17:32:04,245 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-05-27 17:32:04,380 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 17:32:04,398 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-05-27 17:32:04,418 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-05-27 17:32:04,492 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 17:32:04,510 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-27 17:32:04,529 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

2026-05-27 17:32:04,602 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 17:32:04,619 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-27 17:32:04,680 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-27 17:32:04,697 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"
2026-05-27 17:32:04,715 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

2026-05-27 17:32:08,211 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-27 17:32:08,359 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 17:32:08,377 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-05-27 17:32:08,396 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-05-27 17:32:08,480 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-05-27 17:32:08,547 - INFO - 1 prompt is loaded, with the key: query



--- Chunk 1 ---

Foundational Large Language Models & Text Generation
52
February 2025
Using large language models
Prompt engineering and sampling techniques have a strong influence on the performance of 
LLMs. Prompt engineering is the process of designing and refining the text inputs (prompts) 
that you feed into an LLM to achieve desired and relevant outputs. Sampling techniques 
determine the way in which output tokens are chosen and influence the correctness, 
creativity and diversity of the resulting output. We next discuss different variants of prompt 
engineering and sampling techniques as well as touch on some important parameters that 
can have a significant impact on LLM performance.
Prompt engineering 
LLMs are very powerful, but they still need guidance to unleash their full potential. Prompt 
engineering is a critical component in guiding an LLM to yield desired outputs. This might 
include grounding the model to yield factual responses or unleashing the creativity of th

# Lesson 4
## Permanent storage (ChromaDB)

In [30]:
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp protobuf
!pip install chromadb==0.5.5 \
    opentelemetry-api==1.27.0 \
    opentelemetry-sdk==1.27.0 \
    opentelemetry-exporter-otlp==1.27.0 \
    protobuf==4.25.3 -q

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 14.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.8 MB/s eta 0:0

In [31]:
!pip install -U chromadb==0.5.23 llama-index-vector-stores-chroma==0.5.5 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 11.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 66.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [32]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

db = chromadb.PersistentClient(path='./chroma_db')
collection = db.get_or_create_collection('whitepaper')

2026-05-27 17:32:41,792 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-05-27 17:32:42,317 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-05-27 17:32:42,351 - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


**Meaning**
* creates folder chroma_db
* collection/db name = whitepapers
* saved permanently

In [33]:
#connect to llama_index

vector_store = ChromaVectorStore(chroma_collection=collection)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

now llama index writes into chroma

In [34]:
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

2026-05-27 17:32:45,558 - ERROR - Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [35]:
db = chromadb.PersistentClient(path="./chroma_db")

collection = db.get_collection("whitepaper")
print(collection.count())

2026-05-27 17:32:45,570 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


260


## Metadata Filtering

In [36]:
files = set()

for doc in documents:
    files.add(doc.metadata['file_name'])

print(files)
print(len(files))

{'whitepaper_emebddings_vectorstores_v2.pdf', '22365_3_Prompt Engineering_v7 (1).pdf', 'whitepaper 1.pdf', '22365_19_Agents_v8.pdf'}
4


In [37]:
retriever = index.as_retriever(
    similarity_top_k=2,
    fiters={
    'file_name':'22365_19_Agents_v8.pdf'
    }
)

nodes = retriever.retrieve('What are agents?')

for node in nodes:
    print(node.metadata)
    print(node.text[:800])
    print("="*50)

2026-05-27 17:32:45,630 - ERROR - Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'page_label': '5', 'file_name': '22365_19_Agents_v8.pdf'}
Agents
5
February 2025
What is an agent?
In its most fundamental form, a Generative AI agent can be defined as an application that 
attempts to achieve a goal by observing the world and acting upon it using the tools that it 
has at its disposal. Agents are autonomous and can act independently of human intervention, 
especially when provided with proper goals or objectives they are meant to achieve. Agents 
can also be proactive in their approach to reaching their goals. Even in the absence of 
explicit instruction sets from a human, an agent can reason about what it should do next to 
achieve its ultimate goal. While the notion of agents in AI is quite general and powerful, this 
whitepaper focuses on the specific types of agents that Generative AI models are capable of 
building at the t
{'page_label': '40', 'file_name': '22365_19_Agents_v8.pdf'}
Agents
40
February 2025
Summary
In this whitepaper we’ve discussed the foundatio

In [38]:
for node in nodes:
    print(node.score)

0.668149476739516
0.563455225204821


Whenever chunks are retrieved check for:
* Relevance - Does the chunk acctually contain some answerable info
* Completeness - Is the answer split across multiple chunks
* Noise - Does the retrieved chunk contain unrelated surrounding text

# Lesson 5 
## Multi-query retrival
Instead of one query:

    "How do LLMs work?"

Generate multiple related queries:

    "LLM architecture"
    "next token prediction"
    "autoregressive generation"
    "text generation process"

Then retrieve from all of them.
This improves recall.

This is the first time retrieval itself becomes “LLM-assisted.”
* Until now:
embeddings handled retrieval
* Now:
an LLM helps reformulate search queries

In [39]:
#Manual version

retriever = index.as_retriever(
    similarity_top_k=2,
)

queries = [
    "What is prompt engineering?",
    "How are prompts designed?",
    "Prompt optimization techniques",
    "Writing effective prompts for LLMs"
]

all_results = []

for query in queries:
    nodes = retriever.retrieve(query)
    
    print('\nQuery:',query )
    print("="*50)

    for n in nodes:
        print(n.text[:500])
        print("-"*30)

        all_results.append(n.text)


Query: What is prompt engineering?
Foundational Large Language Models & Text Generation
52
February 2025
Using large language models
Prompt engineering and sampling techniques have a strong influence on the performance of 
LLMs. Prompt engineering is the process of designing and refining the text inputs (prompts) 
that you feed into an LLM to achieve desired and relevant outputs. Sampling techniques 
determine the way in which output tokens are chosen and influence the correctness, 
creativity and diversity of the resulting outpu
------------------------------
Prompt Engineering
February 2025
7
When you chat with the Gemini chatbot,1 you basically write prompts, however this 
whitepaper focuses on writing prompts for the Gemini model within Vertex AI or by using  
the API, because by prompting the model directly you will have access to the configuration 
such as temperature etc.
This whitepaper discusses prompt engineering in detail. We will look into the various 
prompting techniques

**Retrieval quality depends heavily on query formulation.**
Now lets automate this process using a llm 

**Goal**

Pipeline becomes:

user query
    → LLM generates alternate queries
        → retrieve for all queries
            → merge results

This is commonly called: query expansion
or multi-query retrieval

In [40]:
!pip install transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 11.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 90.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
ER

In [42]:
from transformers import pipeline

generator = pipeline(
    'text-generation',
    model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    device_map='auto'
)

prompt = """
Generate 3 different search queries related to:
'What is prompt engineering?'

Return only the queries.
"""

response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(response[0]["generated_text"])

2026-05-27 18:18:40,896 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 18:18:40,914 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
2026-05-27 18:18:40,988 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 18:18:41,005 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
2026-05-27 18:18:41,069 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-05-27 18:18:41,131 - INFO - HTTP Request: GET https://huggingface.co/api/models/Tin

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-05-27 18:18:51,907 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 18:18:51,925 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
2026-05-27 18:18:51,947 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

2026-05-27 18:18:52,993 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-05-27 18:18:53,054 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 18:18:53,073 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/special_tokens_map.json "HTTP/1.1 200 OK"
2026-05-27 18:18:53,092 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

2026-05-27 18:18:53,179 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generate 3 different search queries related to:
'What is prompt engineering?'

Return only the queries.

Examples:

1. What is prompt engineering?
2. What is the best way to promote prompt engineering?
3. How can I promote prompt engineering in my company?
4. What are the benefits of prompt engineering?
5. What are the challenges of prompt engineering?
6. How can I improve prompt engineering in my team?
7. What are the key factors for promoting prompt engineering?
8. How can I measure the success of prompt


In [44]:
all_nodes = []

for q in response[0]["generated_text"]:
    nodes = retriever.retrieve(q)
    all_nodes.extend(nodes)